In [ ]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 95.4 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
# paste your FULL code here (same as you gave)
import streamlit as st
import pandas as pd
import numpy as np
import ast
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ─────────────────────────────────────────────
# PAGE CONFIG
# ─────────────────────────────────────────────
st.set_page_config(
    page_title="CineMood · TMDB Recommender",
    page_icon="🎬",
    layout="centered",
    initial_sidebar_state="collapsed",
)

# ─────────────────────────────────────────────
# CUSTOM CSS — cinematic dark theme
# ─────────────────────────────────────────────
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@700;900&family=DM+Sans:wght@300;400;500&display=swap');

:root {
    --gold:   #F5C842;
    --red:    #E5383B;
    --dark:   #0D0D0D;
    --card:   #161616;
    --muted:  #888;
    --border: #2a2a2a;
}
html, body, [class*="css"] {
    font-family: 'DM Sans', sans-serif;
    background-color: var(--dark);
    color: #e8e8e8;
}
#MainMenu, footer, header { visibility: hidden; }

.hero { text-align: center; padding: 2.5rem 1rem 1rem; }
.hero h1 {
    font-family: 'Playfair Display', serif;
    font-size: clamp(2.2rem, 6vw, 3.8rem);
    font-weight: 900;
    letter-spacing: -1px;
    background: linear-gradient(135deg, #F5C842 0%, #E5383B 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    margin-bottom: 0.4rem;
}
.hero p { color: var(--muted); font-size: 1.05rem; font-weight: 300; }
.divider {
    border: none; height: 1px;
    background: linear-gradient(90deg, transparent, var(--border), transparent);
    margin: 2rem 0;
}
.section-label {
    font-size: 0.72rem; font-weight: 500;
    letter-spacing: 2.5px; text-transform: uppercase;
    color: var(--gold); margin-bottom: 0.75rem;
}
.stButton > button {
    background: linear-gradient(135deg, #F5C842, #E5A020);
    color: #0D0D0D; font-family: 'DM Sans', sans-serif;
    font-weight: 700; font-size: 1rem; border: none;
    border-radius: 10px; padding: 0.75rem 2.5rem;
    width: 100%; cursor: pointer; transition: all 0.2s;
}
.stButton > button:hover {
    transform: translateY(-2px);
    box-shadow: 0 6px 24px rgba(245,200,66,0.3);
}

.movie-card {
    background: var(--card); border: 1px solid var(--border);
    border-radius: 16px; padding: 1.6rem 1.8rem;
    margin-bottom: 1.2rem; position: relative; overflow: hidden;
}
.movie-card::before {
    content: ''; position: absolute; top: 0; left: 0;
    width: 4px; height: 100%;
    background: linear-gradient(180deg, #F5C842, #E5383B);
    border-radius: 16px 0 0 16px;
}
.rank-badge {
    display: inline-block;
    background: linear-gradient(135deg, #F5C842, #E5A020);
    color: #0D0D0D; font-weight: 700; font-size: 0.75rem;
    letter-spacing: 1px; padding: 3px 10px; border-radius: 20px;
    margin-bottom: 0.7rem; text-transform: uppercase;
}
.movie-title {
    font-family: 'Playfair Display', serif;
    font-size: 1.45rem; font-weight: 700;
    color: #f0f0f0; margin-bottom: 0.5rem;
}
.movie-meta { display: flex; gap: 0.6rem; flex-wrap: wrap; margin-bottom: 0.9rem; }
.meta-pill {
    background: #222; border: 1px solid var(--border);
    border-radius: 20px; padding: 3px 12px;
    font-size: 0.8rem; color: #bbb;
}
.meta-pill.rating {
    background: rgba(245,200,66,0.12);
    border-color: rgba(245,200,66,0.3);
    color: var(--gold); font-weight: 600;
}
.overview {
    font-size: 0.88rem; color: #aaa;
    line-height: 1.6; margin-bottom: 0.9rem;
    font-style: italic;
}
.explanation {
    background: rgba(229,56,59,0.07);
    border: 1px solid rgba(229,56,59,0.18);
    border-radius: 8px; padding: 0.65rem 0.9rem;
    font-size: 0.85rem; color: #c8c8c8; line-height: 1.55;
}
.explanation span { color: #F5C842; font-weight: 500; }
.no-results { text-align: center; padding: 3rem 1rem; color: var(--muted); }
.upload-box {
    background: #111; border: 2px dashed #333;
    border-radius: 12px; padding: 2rem; text-align: center;
    color: #666; font-size: 0.9rem;
}
.stat-row {
    display: flex; gap: 1.5rem; justify-content: center;
    margin: 1rem 0 1.5rem;
}
.stat { text-align: center; }
.stat-num { font-size: 1.6rem; font-weight: 700; color: var(--gold); }
.stat-lbl { font-size: 0.72rem; color: var(--muted); text-transform: uppercase; letter-spacing: 1px; }
</style>
""", unsafe_allow_html=True)

# ─────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────
MOOD_MAP = {
    "sad":      ["comedy", "feel-good", "romance", "family"],
    "unhappy":  ["comedy", "feel-good", "romance"],
    "crying":   ["comedy", "romance", "heartwarming"],
    "bored":    ["thriller", "action", "crime"],
    "dull":     ["thriller", "action", "mystery"],
    "stressed": ["animation", "family", "light", "fantasy"],
    "anxious":  ["animation", "family", "comedy"],
    "tired":    ["animation", "comedy", "light"],
    "excited":  ["adventure", "science fiction", "action"],
    "happy":    ["adventure", "comedy", "animation"],
    "romantic": ["romance", "drama"],
    "lonely":   ["romance", "drama", "feel-good"],
    "angry":    ["thriller", "action", "crime"],
    "curious":  ["science fiction", "mystery", "thriller"],
    "nostalgic":["drama", "history", "romance"],
    "scared":   ["comedy", "animation", "family"],
}

MOOD_EMOJIS = {
    "sad": "😢", "bored": "😑", "stressed": "😤",
    "excited": "🤩", "romantic": "💕", "happy": "😄",
    "angry": "😠", "curious": "🤔", "lonely": "🥺",
    "nostalgic": "🌅", "scared": "😨", "tired": "😴",
}

# ─────────────────────────────────────────────
# JSON PARSING HELPERS
# ─────────────────────────────────────────────
def parse_json_names(val, limit=None):
    """Extract 'name' fields from a JSON-encoded list of dicts."""
    try:
        items = ast.literal_eval(str(val))
        names = [i["name"] for i in items if "name" in i]
        return names[:limit] if limit else names
    except Exception:
        return []

def parse_director(crew_val):
    try:
        crew = ast.literal_eval(str(crew_val))
        for member in crew:
            if member.get("job") == "Director":
                return member.get("name", "")
        return ""
    except Exception:
        return ""

def parse_cast_names(cast_val, limit=3):
    try:
        cast = ast.literal_eval(str(cast_val))
        return [c["name"] for c in cast[:limit] if "name" in c]
    except Exception:
        return []

# ─────────────────────────────────────────────
# DATA LOADING & PREPROCESSING
# ─────────────────────────────────────────────
@st.cache_data(show_spinner=False)
def load_and_preprocess(movies_path, credits_path=None):
    df = pd.read_csv(movies_path)

    # Merge with credits if provided
    if credits_path:
        credits = pd.read_csv(credits_path)
        # credits has columns: movie_id, title, cast, crew
        # movies has: id, title, ...
        if "movie_id" in credits.columns:
            credits = credits.rename(columns={"movie_id": "id"})
        df = df.merge(credits[["id", "cast", "crew"]], on="id", how="left")

    # ── Parse JSON columns ───────────────────
    df["genres_list"]   = df["genres"].apply(lambda x: parse_json_names(x))
    df["keywords_list"] = df["keywords"].apply(lambda x: parse_json_names(x, limit=10))

    if "cast" in df.columns:
        df["cast_list"] = df["cast"].apply(lambda x: parse_cast_names(x, 3))
    else:
        df["cast_list"] = [[] for _ in range(len(df))]

    if "crew" in df.columns:
        df["director"] = df["crew"].apply(parse_director)
    else:
        df["director"] = ""

    # ── Text normalization ───────────────────
    df["genres_str"]   = df["genres_list"].apply(lambda x: " ".join(x).lower())
    df["keywords_str"] = df["keywords_list"].apply(lambda x: " ".join(x).lower())
    df["cast_str"]     = df["cast_list"].apply(lambda x: " ".join(x).lower())
    df["overview"]     = df["overview"].fillna("").str.lower()

    # ── Release year ─────────────────────────
    if "release_date" in df.columns:
        df["release_year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year.fillna(0).astype(int)
    else:
        df["release_year"] = 0

    # ── Rating ───────────────────────────────
    rating_col = "vote_average" if "vote_average" in df.columns else "rating"
    df["rating"] = pd.to_numeric(df.get(rating_col, 0), errors="coerce").fillna(0.0)

    # votes for weighted rating
    if "vote_count" in df.columns:
        df["vote_count"] = pd.to_numeric(df["vote_count"], errors="coerce").fillna(0)
    else:
        df["vote_count"] = 0

    # Weighted rating: W = (v/(v+m)) * R + (m/(v+m)) * C
    m = df["vote_count"].quantile(0.70)
    C = df["rating"].mean()
    df["weighted_rating"] = (
        (df["vote_count"] / (df["vote_count"] + m)) * df["rating"] +
        (m / (df["vote_count"] + m)) * C
    ).round(2)

    # ── Combined feature column ───────────────
    df["features"] = (
        df["genres_str"] + " " +
        df["keywords_str"] + " " +
        df["overview"] + " " +
        df["cast_str"]
    )

    df["title"] = df["title"].fillna("Unknown")

    return df

# ─────────────────────────────────────────────
# MOOD DETECTION
# ─────────────────────────────────────────────
def detect_mood(text: str):
    text = text.lower()
    for mood in MOOD_MAP:
        if mood in text:
            return mood
    return None

def get_mood_features(mood):
    if mood and mood in MOOD_MAP:
        return MOOD_MAP[mood]
    return MOOD_MAP["bored"]   # default

# ─────────────────────────────────────────────
# RECOMMENDATION ENGINE
# ─────────────────────────────────────────────
def recommend(df, mood_text, genre, year_range, keywords, top_n=3):
    detected_mood = detect_mood(mood_text)
    mood_features = get_mood_features(detected_mood)

    # Build query
    parts = [f.replace("-", " ") for f in mood_features]
    if genre and genre.lower() not in ("any", ""):
        parts.append(genre.lower())
    if keywords.strip():
        parts.extend(keywords.lower().split())

    query = " ".join(parts)

    # Year filter
    filtered = df.copy()
    if year_range:
        y_min, y_max = year_range
        filtered = filtered[
            (filtered["release_year"] >= y_min) &
            (filtered["release_year"] <= y_max) &
            (filtered["release_year"] > 0)
        ]

    if filtered.empty:
        return pd.DataFrame(), detected_mood

    # TF-IDF cosine similarity
    corpus = filtered["features"].tolist() + [query]
    tfidf  = TfidfVectorizer(stop_words="english", max_features=20000)
    matrix = tfidf.fit_transform(corpus)

    scores = cosine_similarity(matrix[-1], matrix[:-1]).flatten()
    filtered = filtered.copy()
    filtered["similarity"] = scores

    # Top 50 by similarity → rank by weighted_rating → top N
    top_similar = filtered.nlargest(50, "similarity")
    top_rated   = top_similar.nlargest(top_n, "weighted_rating")

    return top_rated.reset_index(drop=True), detected_mood

# ─────────────────────────────────────────────
# EXPLANATION
# ─────────────────────────────────────────────
def build_explanation(row, detected_mood, genre, keywords):
    parts = []
    if detected_mood:
        mapped = ", ".join(MOOD_MAP[detected_mood][:3])
        parts.append(f"matches your <span>{detected_mood}</span> mood with {mapped} themes")
    if genre and genre.lower() not in ("any", ""):
        parts.append(f"fits the <span>{genre}</span> genre you selected")
    if keywords.strip():
        kw_list = keywords.strip().split()[:3]
        parts.append(f"aligns with your keywords: <span>{', '.join(kw_list)}</span>")
    if not parts:
        parts.append("is a top-rated, highly popular pick from the TMDB dataset")

    return "Recommended because it " + "; and it ".join(parts) + "."

# ─────────────────────────────────────────────
# MAIN UI
# ─────────────────────────────────────────────
def main():
    # ── Hero ─────────────────────────────────
    st.markdown("""
    <div class="hero">
        <h1>🎬 CineMood</h1>
        <p>Powered by TMDB 5000 · Tell us how you feel, we'll find your film.</p>
    </div>
    <hr class="divider">
    """, unsafe_allow_html=True)

    # ── File Upload ───────────────────────────
    st.markdown('<div class="section-label">📂 Upload TMDB Dataset</div>', unsafe_allow_html=True)

    col_u1, col_u2 = st.columns(2)
    with col_u1:
        movies_file  = st.file_uploader("tmdb_5000_movies.csv", type="csv", key="movies")
    with col_u2:
        credits_file = st.file_uploader("tmdb_5000_credits.csv (optional)", type="csv", key="credits")

    if movies_file is None:
        st.markdown("""
        <div class="upload-box">
            📥 Upload <strong>tmdb_5000_movies.csv</strong> to get started.<br>
            Optionally also upload <strong>tmdb_5000_credits.csv</strong> for cast & director data.<br><br>
            <small>Download from <a href="https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata" target="_blank" style="color:#F5C842">kaggle.com/datasets/tmdb/tmdb-movie-metadata</a></small>
        </div>
        """, unsafe_allow_html=True)
        return

    # ── Load Data ─────────────────────────────
    with st.spinner("📊 Preprocessing TMDB dataset…"):
        try:
            # save to temp paths
            movies_path  = "/tmp/tmdb_movies.csv"
            credits_path = None

            with open(movies_path, "wb") as f:
                f.write(movies_file.read())

            if credits_file:
                credits_path = "/tmp/tmdb_credits.csv"
                with open(credits_path, "wb") as f:
                    f.write(credits_file.read())

            df = load_and_preprocess(movies_path, credits_path)
        except Exception as e:
            st.error(f"❌ Error loading data: {e}")
            return

    # Stats bar
    all_genres = sorted(set(g for gl in df["genres_list"] for g in gl))
    year_min   = int(df[df["release_year"] > 0]["release_year"].min())
    year_max   = int(df[df["release_year"] > 0]["release_year"].max())

    st.markdown(f"""
    <div class="stat-row">
        <div class="stat"><div class="stat-num">{len(df):,}</div><div class="stat-lbl">Movies</div></div>
        <div class="stat"><div class="stat-num">{len(all_genres)}</div><div class="stat-lbl">Genres</div></div>
        <div class="stat"><div class="stat-num">{year_min}–{year_max}</div><div class="stat-lbl">Years</div></div>
        <div class="stat"><div class="stat-num">{df['rating'].max():.1f}</div><div class="stat-lbl">Max Rating</div></div>
    </div>
    """, unsafe_allow_html=True)

    st.markdown('<hr class="divider">', unsafe_allow_html=True)

    # ── Input Fields ──────────────────────────
    col1, col2 = st.columns(2, gap="medium")

    with col1:
        st.markdown('<div class="section-label">😌 Your Mood</div>', unsafe_allow_html=True)
        mood_input = st.text_input(
            "mood", placeholder='e.g. "feeling sad", "super excited"…',
            label_visibility="collapsed",
        )
        st.markdown(
            '<div style="display:flex;flex-wrap:wrap;gap:6px;margin-top:6px">'
            + "".join(f'<span style="background:#1e1e1e;border:1px solid #2a2a2a;border-radius:20px;padding:4px 12px;font-size:0.78rem;color:#aaa">{e} {m}</span>'
                      for m, e in MOOD_EMOJIS.items())
            + "</div>", unsafe_allow_html=True,
        )

    with col2:
        st.markdown('<div class="section-label">🎭 Genre</div>', unsafe_allow_html=True)
        genre_options = ["Any"] + all_genres
        genre_input = st.selectbox("genre", genre_options, label_visibility="collapsed")

    st.markdown("<div style='height:1rem'></div>", unsafe_allow_html=True)
    col3, col4 = st.columns(2, gap="medium")

    with col3:
        st.markdown('<div class="section-label">📅 Release Year Range</div>', unsafe_allow_html=True)
        use_year = st.checkbox("Filter by year", value=False)
        if use_year:
            year_range = st.slider(
                "years", min_value=year_min, max_value=year_max,
                value=(2000, year_max), label_visibility="collapsed",
            )
        else:
            year_range = None

    with col4:
        st.markdown('<div class="section-label">🔑 Keywords</div>', unsafe_allow_html=True)
        keywords_input = st.text_input(
            "keywords", placeholder="e.g. space, time travel, heist…",
            label_visibility="collapsed",
        )

    st.markdown("<div style='height:0.5rem'></div>", unsafe_allow_html=True)
    run = st.button("✨ Recommend Movies")
    st.markdown('<hr class="divider">', unsafe_allow_html=True)

    # ── Results ───────────────────────────────
    if run:
        if not mood_input.strip() and genre_input == "Any" and not keywords_input.strip():
            st.warning("💡 Enter at least a mood, genre, or keyword to get recommendations.")
            return

        with st.spinner("🎞️ Finding your perfect films…"):
            results, detected_mood = recommend(
                df,
                mood_input,
                genre_input if genre_input != "Any" else "",
                year_range,
                keywords_input,
            )

        if results.empty:
            st.markdown("""
            <div class="no-results">
                🎞️ No matches found.<br>
                <small>Try relaxing the year filter or broadening your keywords.</small>
            </div>""", unsafe_allow_html=True)
            return

        # Mood banner
        if detected_mood:
            emoji = MOOD_EMOJIS.get(detected_mood, "🎬")
            st.markdown(
                f'<div style="text-align:center;color:#888;font-size:0.85rem;margin-bottom:1.2rem;">'
                f'Detected mood: <span style="color:#F5C842;font-weight:600;">{emoji} {detected_mood}</span></div>',
                unsafe_allow_html=True,
            )

        rank_labels = ["🥇 Top Pick", "🥈 Second Pick", "🥉 Third Pick"]

        for i, (_, row) in enumerate(results.iterrows()):
            explanation  = build_explanation(row, detected_mood, genre_input, keywords_input)
            genres_disp  = ", ".join(row["genres_list"][:3]) if row["genres_list"] else "N/A"
            cast_disp    = ", ".join(row["cast_list"][:3])   if row["cast_list"] else ""
            director     = row.get("director", "")
            overview_txt = row["overview"].strip().capitalize()[:220]
            if len(row["overview"]) > 220:
                overview_txt += "…"
            stars = "★" * int(round(row["rating"] / 2)) + "☆" * (5 - int(round(row["rating"] / 2)))

            cast_pill    = f'<span class="meta-pill">🎭 {cast_disp}</span>'     if cast_disp else ""
            director_pill= f'<span class="meta-pill">🎬 {director}</span>'       if director  else ""

            st.markdown(f"""
            <div class="movie-card">
                <div class="rank-badge">{rank_labels[i] if i < 3 else f"#{i+1}"}</div>
                <div class="movie-title">{row['title']}</div>
                <div class="movie-meta">
                    <span class="meta-pill">🎞️ {genres_disp}</span>
                    <span class="meta-pill">📅 {row['release_year']}</span>
                    <span class="meta-pill rating">⭐ {row['rating']:.1f} / 10 &nbsp;{stars}</span>
                    <span class="meta-pill rating" style="background:rgba(229,56,59,0.1);border-color:rgba(229,56,59,0.3);color:#E5383B">
                        🏆 Weighted {row['weighted_rating']:.2f}
                    </span>
                    {cast_pill}{director_pill}
                </div>
                {"<div class='overview'>" + overview_txt + "</div>" if overview_txt else ""}
                <div class="explanation">💡 {explanation}</div>
            </div>
            """, unsafe_allow_html=True)
    else:
        st.markdown("""
        <div class="no-results" style="padding:2rem 1rem;">
            🍿 &nbsp;Upload the TMDB CSV files, enter your mood above,<br>
            then hit <strong style="color:#F5C842;">Recommend Movies</strong>
        </div>""", unsafe_allow_html=True)


if __name__ == "__main__":
    main()

Writing app.py


In [ ]:
from pyngrok import ngrok

# 🔑 Add your token here
ngrok.set_auth_token("3B5ELeGFdndPgJctZt90mqOU4Au_2JFRddfnhQByTdgye7PWe")

In [ ]:
from pyngrok import ngrok

# Kill previous tunnels if any
ngrok.kill()

# Start Streamlit
!streamlit run app.py &>/dev/null &

# Create public URL
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://oppositionless-adrian-unswervingly.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
import streamlit, pyngrok

from pyngrok import ngrok

# 🔑 IMPORTANT
ngrok.set_auth_token("3B5ELeGFdndPgJctZt90mqOU4Au_2JFRddfnhQByTdgye7PWe")

!streamlit run app.py &>/dev/null &

public_url = ngrok.connect(8501)
print("Your app is live at:", public_url)

Your app is live at: NgrokTunnel: "https://oppositionless-adrian-unswervingly.ngrok-free.dev" -> "http://localhost:8501"
